In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# File handling
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Project root
project_root = Path("..")

# Dataset folder
data_folder = project_root / "data" / "raw" / "2024" / "results"

# Find all CSV files
csv_files = sorted(data_folder.glob("*.csv"))

print("Dataset folder:", data_folder)
print("Number of CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Dataset folder: ..\data\raw\2024\results
Number of CSV files found: 12
test_result_202401.csv
test_result_202402.csv
test_result_202403.csv
test_result_202404.csv
test_result_202405.csv
test_result_202406.csv
test_result_202407.csv
test_result_202408.csv
test_result_202409.csv
test_result_202410.csv
test_result_202411.csv
test_result_202412.csv


In [3]:
# Select the first CSV file
first_file = csv_files[0]

print("Inspecting:", first_file.name)

# Read only the first 10 rows
df_sample = pd.read_csv(first_file, nrows=10)

df_sample

Inspecting: test_result_202401.csv


,test_id,vehicle_id,test_date,test_class_id,test_type,test_result,test_mileage,postcode_area,make,model,colour,fuel_type,cylinder_capacity,first_use_date,completed_date
0,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
1,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
2,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
3,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
4,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
5,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
6,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
7,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
8,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z
9,886090729,1182957512,2024-01-02,4,NT,P,71614.0,OX,PORSCHE,911,RED,PE,3824,2010-09-01,2024-01-02T10:19:28.000Z


In [4]:
df_sample.columns.tolist()

['test_id',
 'vehicle_id',
 'test_date',
 'test_class_id',
 'test_type',
 'test_result',
 'test_mileage',
 'postcode_area',
 'make',
 'model',
 'colour',
 'fuel_type',
 'cylinder_capacity',
 'first_use_date',
 'completed_date']

In [5]:
# Check the data types of each column
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   test_id            10 non-null     int64  
 1   vehicle_id         10 non-null     int64  
 2   test_date          10 non-null     str    
 3   test_class_id      10 non-null     int64  
 4   test_type          10 non-null     str    
 5   test_result        10 non-null     str    
 6   test_mileage       10 non-null     float64
 7   postcode_area      10 non-null     str    
 8   make               10 non-null     str    
 9   model              10 non-null     int64  
 10  colour             10 non-null     str    
 11  fuel_type          10 non-null     str    
 12  cylinder_capacity  10 non-null     int64  
 13  first_use_date     10 non-null     str    
 14  completed_date     10 non-null     str    
dtypes: float64(1), int64(5), str(9)
memory usage: 1.3 KB


In [6]:
categorical_columns = [
    "test_class_id",
    "test_type",
    "test_result",
    "fuel_type",
    "make",
    "model",
    "colour"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df_sample[column].value_counts(dropna=False))


--- test_class_id ---
test_class_id
4    10
Name: count, dtype: int64

--- test_type ---
test_type
NT    10
Name: count, dtype: int64

--- test_result ---
test_result
P    10
Name: count, dtype: int64

--- fuel_type ---
fuel_type
PE    10
Name: count, dtype: int64

--- make ---
make
PORSCHE    10
Name: count, dtype: int64

--- model ---
model
911    10
Name: count, dtype: int64

--- colour ---
colour
RED    10
Name: count, dtype: int64


In [7]:
# Count important categories across the full January dataset

test_class_counts = {}
test_type_counts = {}
test_result_counts = {}

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    for value, count in chunk["test_class_id"].value_counts().items():
        test_class_counts[value] = test_class_counts.get(value, 0) + count

    for value, count in chunk["test_type"].value_counts().items():
        test_type_counts[value] = test_type_counts.get(value, 0) + count

    for value, count in chunk["test_result"].value_counts().items():
        test_result_counts[value] = test_result_counts.get(value, 0) + count


print("TEST CLASSES")
print(test_class_counts)

print("\nTEST TYPES")
print(test_type_counts)

print("\nTEST RESULTS")
print(test_result_counts)

TEST CLASSES
{'4': 4124645, '7': 128365, '2': 33638, '1': 15095, '5': 5490, '3': 651, 'test_class_id': 11}

TEST TYPES
{'NT': 3586898, 'RT': 720959, 'test_type': 11, 'EI': 18, 'ES': 9}

TEST RESULTS
{'P': 3310945, 'F': 771976, 'PRS': 199767, 'ABR': 22367, 'ABA': 2829, 'test_result': 11}


In [8]:
# Count results for Class 4 initial MOT tests only

class4_nt_results = {}

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    # Keep only Class 4 initial tests
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT")
    ]

    counts = filtered["test_result"].value_counts()

    for value, count in counts.items():
        class4_nt_results[value] = (
            class4_nt_results.get(value, 0) + count
        )

print("CLASS 4 - INITIAL TEST RESULTS")
print(class4_nt_results)

CLASS 4 - INITIAL TEST RESULTS
{'P': 2487630, 'F': 735504, 'PRS': 189097, 'ABR': 17816, 'ABA': 2669}


In [9]:
# Define the Model 1 target mapping

target_mapping = {
    "P": 0,
    "F": 1,
    "PRS": 1
}

print(target_mapping)

{'P': 0, 'F': 1, 'PRS': 1}


In [10]:
# Calculate Model 1 target balance for January

pass_count = class4_nt_results["P"]

fail_count = (
    class4_nt_results["F"] +
    class4_nt_results["PRS"]
)

total_count = pass_count + fail_count

pass_percentage = (pass_count / total_count) * 100
fail_percentage = (fail_count / total_count) * 100

print("MODEL 1 TARGET BALANCE")
print(f"Pass (0): {pass_count:,} ({pass_percentage:.2f}%)")
print(f"Fail (1): {fail_count:,} ({fail_percentage:.2f}%)")
print(f"Total usable records: {total_count:,}")

MODEL 1 TARGET BALANCE
Pass (0): 2,487,630 (72.90%)
Fail (1): 924,601 (27.10%)
Total usable records: 3,412,231


In [11]:
# Columns we want to investigate
columns_to_check = [
    "test_class_id",
    "test_type",
    "test_result",
    "test_mileage",
    "postcode_area",
    "make",
    "model",
    "colour",
    "fuel_type",
    "cylinder_capacity",
    "first_use_date"
]

missing_counts = {column: 0 for column in columns_to_check}
total_rows = 0

for chunk in pd.read_csv(
    first_file,
    usecols=columns_to_check,
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    # Keep only records relevant to Model 1
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT") &
        (chunk["test_result"].isin(["P", "F", "PRS"]))
    ]

    total_rows += len(filtered)

    for column in columns_to_check:
        missing_counts[column] += filtered[column].isna().sum()


missing_summary = pd.DataFrame({
    "Missing Count": missing_counts
})

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / total_rows * 100
).round(2)

print("Total records checked:", f"{total_rows:,}")

missing_summary.sort_values(
    "Missing %",
    ascending=False
)

Total records checked: 3,412,231


,Missing Count,Missing %
cylinder_capacity,30561,0.90
test_mileage,7640,0.22
test_class_id,0,0.00
test_result,0,0.00
test_type,0,0.00
make,0,0.00
postcode_area,0,0.00
model,2,0.00
colour,0,0.00
fuel_type,0,0.00


In [12]:
# Check first_use_date missing values

first_use_missing = 0
total_rows = 0

for chunk in pd.read_csv(
    first_file,
    usecols=["test_class_id", "test_type", "test_result", "first_use_date"],
    dtype=str,
    chunksize=200_000,
    on_bad_lines="skip"
):
    filtered = chunk[
        (chunk["test_class_id"] == "4") &
        (chunk["test_type"] == "NT") &
        (chunk["test_result"].isin(["P", "F", "PRS"]))
    ]

    total_rows += len(filtered)
    first_use_missing += filtered["first_use_date"].isna().sum()

print("Total records:", f"{total_rows:,}")
print("Missing first_use_date:", f"{first_use_missing:,}")
print(
    "Missing percentage:",
    round(first_use_missing / total_rows * 100, 2),
    "%"
)

Total records: 3,412,231
Missing first_use_date: 0
Missing percentage: 0.0 %
